# Cross-Site Comparison

Compare question-production trajectories across Stack Overflow and selected Stack Exchange communities to determine whether the contraction is site-specific or network-wide.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        q=p/'stack_exchange_analysis'
        if (p/'database').exists(): return p
        if (q/'database').exists(): return q
    raise FileNotFoundError
PROJECT_ROOT=root(); DATA_DIR=PROJECT_ROOT/'database'; ANALYSIS_DIR=PROJECT_ROOT/'analysis'; ANALYSIS_DIR.mkdir(exist_ok=True); sys.path.insert(0,str(PROJECT_ROOT/'src'))
from analysis_utils import read_csv_flexible, drop_incomplete_last_period, save_figure

df=read_csv_flexible(DATA_DIR/'new-questions-per-day-by-sites-since-2008.csv'); df['Day']=pd.to_datetime(df['Day'],errors='coerce'); df=df.dropna(subset=['Day'])
site_cols=[c for c in df.columns if c.endswith('Questions') and c!='Day']
for c in site_cols: df[c]=pd.to_numeric(df[c],errors='coerce')
monthly=df.set_index('Day')[site_cols].resample('MS').sum(min_count=1); monthly=drop_incomplete_last_period(monthly,'M')
print(f'{len(site_cols)} sites; {len(monthly)} monthly observations')

## Normalized trajectories

In [ ]:
def normalize(s):
    pos=s[s>0].dropna(); baseline=pos.iloc[:12].mean() if len(pos)>=6 else np.nan; return 100*s/baseline if pd.notna(baseline) and baseline>0 else s*np.nan
idx=monthly.apply(normalize)
selected=[c for c in ['StackOverflowQuestions','MathematicsQuestions','PhysicsQuestions','StatisticsQuestions','ComputerScienceQuestions','DataScienceQuestions','TeXQuestions','AiQuestions','GenAIQuestions'] if c in idx]
fig,ax=plt.subplots(figsize=(12,6))
for c in selected: ax.plot(idx.index,idx[c],label=c.replace('Questions',''),lw=1.2)
ax.set_yscale('log'); ax.set(title='Normalized monthly question activity',xlabel='Month',ylabel='Index: site-specific early baseline = 100'); ax.legend(ncol=3,fontsize=8); ax.grid(alpha=.2); save_figure(fig,ANALYSIS_DIR/'cross_site_normalized.png'); plt.show()

## Relative decline and co-movement

In [ ]:
relative=monthly.divide(monthly.max(axis=0),axis=1); latest=relative.dropna(how='all').iloc[-1].sort_values(); display(latest.rename('latest_fraction_of_peak').to_frame())
growth=monthly.replace(0,np.nan).pct_change(12); corr=growth.corr(min_periods=24)
if 'StackOverflowQuestions' in corr: display(corr['StackOverflowQuestions'].drop('StackOverflowQuestions',errors='ignore').sort_values().rename('correlation_with_stackoverflow_yoy').to_frame())

## PCA of post-2018 trajectories

In [ ]:
recent=monthly.loc[monthly.index>='2018-01-01']; usable=[c for c in site_cols if recent[c].notna().sum()>=48 and (recent[c]>0).sum()>=36]
X=np.log1p(recent[usable]).interpolate(limit_direction='both'); coords=PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X.T)); pca=pd.DataFrame(coords,index=usable,columns=['PC1','PC2'])
fig,ax=plt.subplots(figsize=(8,6)); ax.scatter(pca.PC1,pca.PC2)
for site,row in pca.iterrows(): ax.annotate(site.replace('Questions',''),(row.PC1,row.PC2),fontsize=7)
ax.set(title='PCA of site activity trajectories (2018+)',xlabel='PC1',ylabel='PC2'); ax.grid(alpha=.2); save_figure(fig,ANALYSIS_DIR/'cross_site_pca.png'); plt.show()

## Takeaways
Use the normalized trajectories and co-movement diagnostics to distinguish Stack Overflow-specific decline from broader Stack Exchange contraction. No causal interpretation is imposed.